# Logistics Delay Prediction – Modeling

## Objective
Predict the probability of delivery delays using the Olist dataset.
The target variable is `is_late`, where 1 indicates a delayed order.
Models return **probabilities** via `predict_proba()` rather than binary outputs.

## Plan
1. Baseline model (DummyClassifier)
2. Categorical encoding
3. Train/test split (stratified)
4. Model training: Random Forest, LightGBM, CatBoost
5. Evaluation
6. Hyperparameter tuning (Optuna)
7. Interpretability (SHAP)

## Metrics
Given the class imbalance (~8% late orders), the following metrics are prioritized:
- ROC-AUC
- F1-Score
- Recall

## Stack
- scikit-learn
- LightGBM
- CatBoost
- Optuna
- SHAP

In [11]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, recall_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold

import lightgbm as lgb
import catboost as cb

# ── Hyperparameter tuning ─────────────────────────────────────────────────────
import optuna

# ── Interpretability ──────────────────────────────────────────────────────────
import shap

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_parquet("consolidated_logistics_base.parquet")
print(f"Dataset loaded: {df.shape}")

Dataset loaded: (110196, 67)


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110196 entries, 0 to 110195
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   seller_id                    110196 non-null  str    
 1   customer_state               110196 non-null  str    
 2   review_score                 109369 non-null  float64
 3   product_category_name        108659 non-null  str    
 4   product_weight_g             110178 non-null  float64
 5   payment_type_main            110193 non-null  str    
 6   seller_geo_city              109947 non-null  str    
 7   seller_geo_state             109947 non-null  str    
 8   customer_geo_city            109908 non-null  str    
 9   customer_geo_state           109908 non-null  str    
 10  seller_customer_distance_km  109660 non-null  float64
 11  is_late                      110196 non-null  Int8   
 12  product_volume_cm3           110178 non-null  float64
 13  is_heavy_p

In [13]:
df.head()

,seller_id,customer_state,review_score,product_category_name,product_weight_g,payment_type_main,seller_geo_city,seller_geo_state,customer_geo_city,customer_geo_state,...,bert_svd_41,bert_svd_42,bert_svd_43,bert_svd_44,bert_svd_45,bert_svd_46,bert_svd_47,bert_svd_48,bert_svd_49,estimated_delivery_dow
0,3504c0cb71d7fa48d967e0e4c94d59d9,SP,4.0,utilidades_domesticas,500.0,voucher,maua,SP,sao paulo,SP,...,-0.098109,0.146767,0.090446,0.280138,-0.106301,-0.437916,0.755499,0.160171,-0.121071,2
1,289cdb325fb7e7f891c38608bf9e0962,BA,4.0,perfumaria,400.0,boleto,belo horizonte,MG,barreiras,BA,...,0.058410,-0.147710,-0.116853,-0.269182,-0.241428,-0.200264,0.136254,-0.166536,0.043802,0
2,4869f7a5dfa277a7dca6462dcf3b52b2,GO,5.0,automotivo,420.0,credit_card,guariba,SP,vianopolis,GO,...,-0.001168,0.001993,0.000650,0.000121,-0.002988,0.000843,-0.000631,-0.002070,-0.001148,1
3,66922902710d126a0e7d26b0e3805106,RN,5.0,pet_shop,450.0,credit_card,belo horizonte,MG,sao goncalo do amarante,RN,...,0.052156,-0.085034,0.380572,0.057153,0.047215,0.005328,-0.367307,-0.231086,0.072944,4
4,2c9e548be18521d1c43cde1c582c6de8,SP,5.0,papelaria,250.0,credit_card,mogi das cruzes,SP,santo andre,SP,...,-0.001168,0.001993,0.000650,0.000121,-0.002988,0.000843,-0.000631,-0.002070,-0.001148,0


In [14]:
# Check target distribution
df["is_late"].value_counts(normalize=True).round(3)

is_late
0    0.921
1    0.079
Name: proportion, dtype: Float64

## Initial Inspection – Main Findings

- Dataset loaded with **110,196 rows** and **67 columns**, consuming ~41 MB in memory.
- Target variable `is_late` is highly imbalanced: **92% on time vs. 8% late** — requires `class_weight='balanced'` or `scale_pos_weight` during training.
- **8 categorical columns** (dtype `str`) require encoding before modeling: `customer_state`, `seller_geo_state`, `customer_geo_state`, `seller_geo_city`, `customer_geo_city`, `product_category_name`, `payment_type_main`, and `seller_id`.
- **50 BERT-SVD features** (float32) are already numeric and ready for modeling.
- A small number of nulls were identified in `review_score`, `product_category_name`, `product_weight_g`, and geo columns — to be handled before split.
- No duplicate columns or unexpected dtypes detected.

In [15]:
categorical_cols = [
    "customer_state", "seller_geo_state", "customer_geo_state",
    "seller_geo_city", "customer_geo_city", "product_category_name",
    "payment_type_main"
]

df = df.drop(columns=["seller_id"])

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[categorical_cols] = encoder.fit_transform(df[categorical_cols])

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110196 entries, 0 to 110195
Data columns (total 66 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   customer_state               110196 non-null  float64
 1   review_score                 109369 non-null  float64
 2   product_category_name        108659 non-null  float64
 3   product_weight_g             110178 non-null  float64
 4   payment_type_main            110193 non-null  float64
 5   seller_geo_city              109947 non-null  float64
 6   seller_geo_state             109947 non-null  float64
 7   customer_geo_city            109908 non-null  float64
 8   customer_geo_state           109908 non-null  float64
 9   seller_customer_distance_km  109660 non-null  float64
 10  is_late                      110196 non-null  Int8   
 11  product_volume_cm3           110178 non-null  float64
 12  is_heavy_product             110196 non-null  Int8   
 13  is_bulky_p

In [17]:
(df["customer_state"] == df["customer_geo_state"]).mean()

np.float64(0.9973864750081672)

## Initial Inspection – Main Findings

- Dataset loaded with **110,196 rows** and **67 columns**, consuming ~41 MB in memory.
- Target variable `is_late` is highly imbalanced: **92% on time vs. 8% late** — requires `class_weight='balanced'` or `scale_pos_weight` during training.
- `customer_geo_state` and `seller_geo_state` dropped — 99.7% identical to `customer_state` and `seller_geo_state` respectively.
- `seller_id` dropped for modeling — high cardinality; reserved for Power BI dashboard.
- **50 BERT-SVD features** (float32) already numeric and ready for modeling.
- Small number of nulls identified in `review_score`, `product_category_name`, `product_weight_g` and geo columns — to be handled before split.

## Encoding Strategy

| Column | Encoding | Rationale |
|---|---|---|
| customer_state | One-Hot | Low cardinality (27 states). No implicit order. |
| seller_geo_city | Target Encoding (k-fold) | High cardinality. Captures predictive signal without dimensionality explosion. |
| customer_geo_city | Target Encoding (k-fold) | Same as above. |
| product_category_name | Target Encoding (k-fold) | Medium-high cardinality (~70 categories). More robust than one-hot. |
| payment_type_main | One-Hot | Very low cardinality (4–5 types). No implicit order. |

In [19]:
# ── Drop redundant and high-cardinality columns ───────────────────────────────
df = df.drop(columns=["customer_geo_state", "seller_geo_state"])

# ── One-Hot Encoding ──────────────────────────────────────────────────────────
ohe_cols = ["customer_state", "payment_type_main"]
df = pd.get_dummies(df, columns=ohe_cols, drop_first=False, dtype=np.int8)

# ── Target Encoding (k-fold) ──────────────────────────────────────────────────
target_enc_cols = ["seller_geo_city", "customer_geo_city", "product_category_name"]
target = df["is_late"].astype(float)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in target_enc_cols:
    df[col + "_te"] = np.nan
    for train_idx, val_idx in kf.split(df):
        mean_enc = target.iloc[train_idx].groupby(df[col].iloc[train_idx]).mean()
        df.loc[df.index[val_idx], col + "_te"] = df[col].iloc[val_idx].map(mean_enc)
    global_mean = target.mean()
    df[col + "_te"] = df[col + "_te"].fillna(global_mean)
    df = df.drop(columns=[col])

print(df.shape)
df.head(3)

(110196, 93)


,review_score,product_weight_g,seller_customer_distance_km,is_late,product_volume_cm3,is_heavy_product,is_bulky_product,has_comment,bert_svd_0,bert_svd_1,...,customer_state_24.0,customer_state_25.0,customer_state_26.0,payment_type_main_0.0,payment_type_main_1.0,payment_type_main_2.0,payment_type_main_3.0,seller_geo_city_te,customer_geo_city_te,product_category_name_te
0,4.0,500.0,18.566632,0,1976.0,0,0,1,-8.512578,-5.002414,...,0,1,0,0,0,0,1,0.020661,0.062273,0.065185
1,4.0,400.0,847.437333,0,4693.0,0,0,1,-8.178789,-4.592004,...,0,0,0,1,0,0,0,0.046867,0.068182,0.076723
2,5.0,420.0,512.100044,0,9576.0,0,0,0,-9.135558,2.350698,...,0,0,0,0,1,0,0,0.116558,0.000000,0.085460
